# Signal Peptide Efficiency Prediction; reproducibility-revised

> **Headline (reproducible):** open-weight vector-NN ensemble (ProtT5 + ESM2-650M + ProtBERT) = **MSE 0.981 ± 0.005**, ~20% over the Grasso RF baseline (1.22).
> All numbers are reported as mean ± std because this architecture is high-variance (~±0.05 per single model). Ginkgo AA-0 (~0.96) is the best single embedding but its API is discontinued and it is unreproducible. See `docs/reproducibility_findings.md`.


# Signal Peptide Efficiency Prediction; Full Reproducibility Notebook

This notebook runs all analysis scripts in sequence, reproducing every result in the manuscript.

**Reproducible headline:** open-weight vector-NN ensemble (ProtT5 + ESM2-650M + ProtBERT) = **MSE 0.981 ± 0.005**, ~20% over the Grasso RF baseline (1.22).

### Prerequisites

1. Install dependencies: `pip install -r requirements.txt`
2. Install PyTorch and Transformers: `pip install torch transformers sentencepiece` (for Scripts 00 and 21; open-weight embedding generation)
3. Place all data files in the `data/` directory (see README for details)
4. Ensure you have sufficient disk space for results and figures

### Runtime Estimates

| Script | Estimated Runtime |
|--------|------------------|
| 00 Generate design embeddings | ~30–60 min (CPU) / ~5–10 min (GPU) |
| 01 Baseline reproduction | ~1 min |
| 02 RF hyperparameter search | ~2–8 hrs |
| 03 NN regression search | ~1–2 hrs |
| 04 Final comparison chart | ~10 sec |
| 05 Cross-dataset generalization | ~30 min |
| 06 Vector regression | ~1–2 hrs |
| 07 Design task evaluation | ~10 min |
| 08 Bootstrap CIs | ~30–40 min |
| 09 Vector architecture search | ~2–3 hrs |
| 10 Vector ensemble optimization | ~5 hrs |
| 11 Dropout validation | ~30 min |
| 12 Bimodal histogram | ~1 min |
| 13 Cross-dataset fine-tuning | ~30–60 min |
| save_best_model | ~5 min |
| 14 ReLU² vs LeakyReLU comparison | ~12 min |
| 15 Regression vs classification | ~25 min |
| 16 Standardize parquet schema | ~2 min |
| 17 Gene-stratified evaluation | ~60–90 min |
| 18 Linear baselines | ~30–60 min |
| 19 Ablation controls | ~30–45 min |

Total: approximately 17–29 hours on CPU.

## Script 01: Baseline Reproduction

Reproduces the Grasso et al. random forest baseline using the exact published hyperparameters (75 trees, max depth 25). Expected test MSE: ~1.193 (within 2.2% of the reported 1.22).

In [ ]:
%%time
!python3 -u scripts/01_grasso_reproduction.py

## Script 02: RF Hyperparameter Search

Performs randomized hyperparameter search (100 iterations, 5-fold CV) independently for each of the four feature types: PhysChem, ESM2-650M, ESM2-3B, and Ginkgo-AA0.

**Warning:** This is one of the longest-running scripts (~2–8 hours).

In [ ]:
%%time
!python3 -u scripts/02_rf_hyperparameter_search.py

## Script 03: NN Regression Search

Searches 40 neural network configurations per feature type with dimension-aware architecture grids. Uses 80/20 train/validation split with early stopping. Best result: Ginkgo-AA0 NN with MSE 1.050.

In [ ]:
%%time
!python3 -u scripts/03_nn_regression.py

## Script 04: Final Comparison Chart

Generates a grouped bar chart comparing all 9 scalar model–feature combinations (Table 1 / Figure 2 in the paper). Quick script that reads saved results from Scripts 01–03.

In [ ]:
%%time
!python3 -u scripts/04_final_comparison.py

## Script 05: Cross-Dataset Generalization

Evaluates the best ESM2-650M RF and 5-seed NN ensemble on four external signal peptide datasets (Wu, Xue, Zhang-P43, Zhang-PglVM). Tests whether models trained on Grasso data generalize across organisms and experimental systems.

In [ ]:
%%time
!python3 -u scripts/05_cross_dataset_generalization.py

## Script 06: Vector Regression

Predicts 10-dimensional bin probability distributions (softmax output) instead of scalar WA. Trains 5-seed ensembles with both cross-entropy and focal loss for all three PLM embedding types. Best result: Ginkgo-AA0 + focal loss, MSE 1.001.

In [ ]:
%%time
!python3 -u scripts/06_vector_regression.py

## Script 00: Generate Design Embeddings

Generates ESM2-650M mean-pooled embeddings for 4,911 unique designed signal peptide variants. This is a one-time prerequisite for the ESM2-650M evaluation in Script 07.

**Warning:** This requires `torch` and `transformers` packages (`pip install torch transformers`). Runtime is ~30–60 minutes on CPU or ~5–10 minutes on GPU.

In [ ]:
%%time
!python3 -u scripts/00_generate_design_embeddings.py

## Script 07: Design Task Evaluation

Tests whether models can predict which designed mutations improve or worsen signal peptide secretion efficiency. Evaluates RF and NN models using both physicochemical features (156d) and ESM2-650M embeddings (1280d, from Script 00) on 4,832 designed variants across 131 genes.

In [ ]:
%%time
!python3 -u scripts/07_design_task_evaluation.py

## Script 08: Bootstrap Confidence Intervals

Computes 95% bootstrap confidence intervals (10,000 resamples) for all 16 models. Produces a forest plot showing MSE point estimates with CI whiskers.

In [ ]:
%%time
!python3 -u scripts/08_bootstrap_ci.py

## Script 09: Vector Architecture Search

Full-data vector regression with no validation split. Explores 3-layer and 4-layer architectures, cosine annealing, and increased ensemble sizes. Identifies (256, 256, 128) as the best architecture with 5-seed ensemble MSE 0.973.

In [ ]:
%%time
!python3 -u scripts/09_vector_architecture_search.py

## Script 10: Vector Ensemble Optimization

Final optimization pass on the best architecture. Tunes dropout rate (0.15–0.35), tests 20-seed ensembles, and mixed-architecture ensembles. Over independent retrains, the (256,256,128)/dropout-0.35 configuration gives MSE 0.957 ± 0.009.

**Warning:** This is the longest-running script (~5 hours on CPU).

In [ ]:
%%time
!python3 -u scripts/10_vector_ensemble_optimization.py

## Script 11: Dropout Validation

Validates the dropout selection from Script 10 using an independent 80/20 train/validation split. Validation sweep over dropout 0.15–0.35: the values are statistically indistinguishable and the validation-optimal is 0.20. Dropout is selected on the validation split rather than the test set; see docs/reproducibility_findings.md.

In [ ]:
%%time
!python3 -u scripts/11_dropout_validation.py

## Script 12: Bimodal Histogram

Generates a motivating figure showing examples where the weighted average (WA) falls between two peaks in the experimental bin probability distribution. This illustrates why predicting full 10-bin distributions (vector regression) is more informative than predicting scalar WA alone.

In [ ]:
%%time
!python3 -u scripts/12_bimodal_histogram.py

## Script 13: Cross-Dataset Fine-Tuning

Fine-tunes the best vector regression ensemble on four external signal peptide datasets using 5-fold cross-validation with 5 seeds per fold. Freezes the pretrained hidden layers and replaces the output head with a task-specific layer.

**Warning:** Runtime ~30-60 minutes on CPU.

In [ ]:
%%time
!python3 -u scripts/13_cross_dataset_finetuning.py

## Save Best Model Weights

Trains the best 5-seed vector NN ensemble (Script 10 config: 256,256,128, dropout=0.35, focal loss) and saves model weights, scaler, and config to `models/`. Validates round-trip loading.

In [ ]:
%%time
!python3 -u scripts/save_best_model.py

## Script 14: ReLU-Squared vs LeakyReLU Comparison

Tests whether ReLU²(x) = max(0,x)² produces sparser and more accurate activations than LeakyReLU for sparse bin distributions. Compares 5-seed ensembles with bootstrap CIs and sparsity analysis.

In [ ]:
%%time
!python3 -u scripts/14_relu_squared_comparison.py

## Script 15: Regression vs Classification Comparison

Compares 5 output formulations (10-bin focal, 10-bin CCE, 5-class, 3-class, binary) on identical architecture and data. The CCE control isolates the effect of output formulation from loss function. All approaches use 5-seed ensembles with bootstrap CIs.

In [ ]:
%%time
!python3 -u scripts/15_regression_vs_classification.py

## Script 16: Standardize Parquet Schema

Consolidates all 7 datasets (Grasso train/test/design + 4 external) into a unified schema with consistent column names. Produces per-dataset and combined parquet files in `data/unified/`.

In [ ]:
%%time
!python3 -u scripts/16_standardize_parquet_schema.py

## Script 17: Gene-Stratified Evaluation

Leave-one-gene-out cross-validation to test whether the model generalizes across genes. 132 of 134 genes overlap between train and test sets (98.5%). Uses 1 seed per fold with bootstrap CIs on aggregated predictions.

**Warning:** Runtime ~60-90 minutes on CPU.

In [ ]:
%%time
!python3 -u scripts/17_gene_stratified_evaluation.py

## Script 18: Linear Baselines

Establishes how much predictive power comes from PLM embeddings vs the MLP head. Compares linear probe (Input→softmax), Ridge regression, and XGBoost against the full NN on Ginkgo-AA0 embeddings.

**Warning:** Runtime ~30-60 minutes on CPU.

In [ ]:
%%time
!python3 -u scripts/18_linear_baseline.py

## Script 19: Ablation Controls

Decomposes the reproducible 0.957 ± 0.009 result into component contributions, and decomposes the LOGO gap into gene-identity leakage vs CV variance.


In [ ]:
%%time
!python3 -u scripts/19_ablation_controls.py

## Summary

Outputs: `results/` (JSON/CSV), `figures/` (300-DPI PNG), `docs/reproducibility_findings.md` (audit), `paper/` (manuscript).

### Headline results (reproducible, mean ± std over retrains)

| Result | Test MSE | Reproducible? |
|---|---|---|
| Grasso RF baseline (Script 01) | 1.22 |; |
| Best scalar NN, ESM2-650M (Script 03) | ~1.05 | open |
| Vector NN, single open embedding (ProtT5) | 1.017 | open |
| **Open-weight ensemble ProtT5+ESM2+ProtBERT (Script 21)** | **0.981 ± 0.005** | **open** |
| Ginkgo AA-0 vector NN (best single) | 0.959 ± 0.014 | ❌ API discontinued |
| Leave-one-gene-out CV (Script 17) | 2.19 (Spearman 0.72) | realistic generalization |
| Cross-dataset fine-tuning, Wu binary (Script 13) | pooled ρ −0.227→+0.341 [0.131,0.536] | significant |
| Design task ±1 WA (Script `design_parity_fig4e`) | ~41% (Grasso Fig 4e: 73%) |; |

The reproducible open-weight ensemble (~0.98) improves ~20% over the Grasso baseline and recovers the discontinued AA-0 (~0.96) to within ~0.02. All numbers are reported as mean ± std because this architecture is high-variance (~±0.05 per single model). See `docs/reproducibility_findings.md`.
